# 02 – FrogML Log Model & AI-BOM

Use the FrogML SDK to log a model to JFrog Artifactory. When you log a model, Artifactory automatically generates an **AI-BOM** (AI Bill of Materials) and **SBOM**, providing traceability for compliance and auditing.

**Prerequisites:** Set `JF_URL`, `JF_ACCESS_TOKEN`, and ensure you have a HuggingFaceML or generic repository in Artifactory.

In [ ]:
# Load .env if present
import os
from pathlib import Path
from datetime import datetime

try:
    env_file = Path(__file__).parent / ".env"
except NameError:
    env_file = Path(".env")
if not env_file.exists():
    env_file = Path("../.env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

jf_url = os.environ.get("JF_URL")
jf_token = os.environ.get("JF_ACCESS_TOKEN")
print(f"JF_URL: {jf_url or '(not set)'}")
print(f"JF_ACCESS_TOKEN: {'***' if jf_token else '(not set)'}")

## Load a small HuggingFace model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "distilbert-base-uncased-finetuned-sst-2-english"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
print("Model loaded.")

## Log model to Artifactory (creates AI-BOM)

`frogml.huggingface.log_model` uploads the model to Artifactory. Artifactory generates the ML-BOM and SBOM automatically. The model appears in the JFrog AI Catalog.

In [ ]:
import frogml

if not jf_url or not jf_token:
    print("Skipping log_model: JF_URL and JF_ACCESS_TOKEN required.")
    print("Set them in .env or environment to log the model to Artifactory.")
else:
    repository = os.environ.get("JF_HF_REPO", "hf-local")
    model_name = "sentiment-demo"
    version = datetime.now().strftime("%Y%m%d-%H%M%S")

    try:
        frogml.huggingface.log_model(
            model=model,
            tokenizer=tokenizer,
            repository=repository,
            model_name=model_name,
            version=version,
            properties={"stage": "demo", "source": "notebook"}
        )
        print(f"Model logged to {repository}/{model_name}:{version}")
        print("Check Artifactory and AI Catalog for the AI-BOM.")
    except Exception as e:
        print(f"Error: {e}")
        print("Ensure the repository exists and you have write access.")

## Alternative: Log a custom model file (.pkl)

For custom models (e.g. scikit-learn pickle files), use `frogml.files.log_model`:

In [ ]:
# Example for a .pkl file (uncomment and adapt if you have one):
# frogml.files.log_model(
#     source_path="model.pkl",
#     repository="generic-local",
#     model_name="my-custom-model",
#     version=datetime.now().strftime("%Y%m%d-%H%M%S"),
#     properties={"stage": "test"}
# )